In [1]:
import json
import requests
from time import sleep
from pathlib import Path
# from collections import Counter


In [ ]:
root_path = Path(__file__).parent


## Statistic informationm

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


### From semantic scholar

In [ ]:
semantic_scholar_snapshot_path = root_path.joinpath('data/semantic_data')


In [ ]:
all_semantic_scholar_conferences = list(semantic_scholar_snapshot_path.glob('*.jsonl'))


In [ ]:
all_semantic_scholar_conferences_abbr = [all_conference.stem.split('_')[-1] for all_conference in all_semantic_scholar_conferences]


In [ ]:
all_semantic_scholar_conferences_abbr


In [ ]:
semantic_scholar_res = {}

for each_conference in all_semantic_scholar_conferences:
    with open(each_conference, 'r') as f:
        papers = [json.loads(line) for line in f]

    for paper in papers:
        published_year = paper.get('published_year', None)
        venue = paper.get('venue', None)
        abbr = paper.get('abbr', None)
        
        if any([published_year is None, venue is None, abbr is None]):
            continue

        if abbr not in semantic_scholar_res:
            semantic_scholar_res[abbr] = {}
        if 'published_year' not in semantic_scholar_res[abbr]:
            semantic_scholar_res[abbr]['published_year'] = {}
        if published_year not in semantic_scholar_res[abbr]['published_year']:
            semantic_scholar_res[abbr]['published_year'][published_year] = 0

        if 'sum' not in semantic_scholar_res[abbr]:
            semantic_scholar_res[abbr]['sum'] = 0

        semantic_scholar_res[abbr]['published_year'][published_year] += 1
        semantic_scholar_res[abbr]['sum'] += 1


In [ ]:
for abbr, data in semantic_scholar_res.items():
    published_years = data.get('published_year', {})
    sorted_published_years = dict(sorted(published_years.items()))
    semantic_scholar_res[abbr]['published_year'] = sorted_published_years

print(json.dumps(semantic_scholar_res, indent=4, ensure_ascii=False))


Citation statitistic

In [ ]:
semantic_scholar_citation_stats = {}

for each_conference in all_semantic_scholar_conferences:
    with open(each_conference, 'r') as f:
        papers = [json.loads(line) for line in f]

    each_venue_citation_stats = []
    for paper in papers:
        published_year = paper.get('published_year', None)
        venue = paper.get('venue', None)
        abbr = paper.get('abbr', None)
        citation_count = paper.get('citation_count', None)
        
        if any([published_year is None, venue is None, abbr is None]):
            continue

        if citation_count is None:
            continue

        each_venue_citation_stats.append(citation_count)

    semantic_scholar_citation_stats[abbr] = pd.DataFrame(each_venue_citation_stats, columns=['citation_count'])


In [ ]:
all_papers_count = 0

for i in semantic_scholar_citation_stats:
    count = len(semantic_scholar_citation_stats[i])
    print(f"Venue: {i},\nTotal number of papers: {count}\tpapers")
    print(semantic_scholar_citation_stats[i].describe())
    print()

    all_papers_count += count


In [ ]:
all_papers_count


### Filter unrelated topic

Create a single file of collected papers

In [ ]:
all_semantic_scholar_conferences


In [ ]:
venue_variants_name = {
    'jss': [
        'Journal of Systems and Software',
    ],
    'saner': [
        'International Conference on Software Analysis, Evolution and Reengineering',
        'International Conference on Software Analysis, Evolution, and Reengineering',
        'IEEE International Conference on Software Analysis, Evolution and Reengineering',
        'IEEE International Conference on Software Analysis, Evolution, and Reengineering',
        'SANER',
    ],
    'fse': [
        'Foundations of Software Engineering',
        'ESEC/FSE',
        'ESEC/SIGSOFT FSE',
        'Proc. ACM Softw. Eng.',
        'ACM Joint European Software Engineering Conference and Symposium on the Foundations of Software Engineering',
        'Proceedings of the ACM Joint European Software Engineering Conference and Symposium on the Foundations of Software Engineering',
        'FSE',
    ],
    'emse': [
        'Empirical Software Engineering',
    ],
    'icse': [
        'International Conference on Software Engineering',
        'Proceedings of the 44th International Conference on Software Engineering',
        'ICSE',
    ],
    'tosem': [
        'ACM Transactions on Software Engineering and Methodology',
        'Transactions on Software Engineering and Methodology',
        'TOSEM',
    ],
    'tse': [
        'IEEE Transactions on Software Engineering',
        'Transactions on Software Engineering',
        'TSE',
    ],
    'ase': [
        'International Conference on Automated Software Engineering',
        'IEEE/ACM International Conference on Automated Software Engineering',
        '2023 38th IEEE/ACM International Conference on Automated Software Engineering (ASE)',
        'ASE',
    ],
    'icsme': [
        'International Conference on Software Maintenance and Evolution',
        'IEEE International Conference on Software Maintenance and Evolution',
        'ICSME',
    ],
    'msr': [
        'Mining Software Repositories',
        'International Conference on Mining Software Repositories',
        'IEEE Working Conference on Mining Software Repositories',
        'MSR',
    ]
}


In [ ]:
filtered_papers_by_title_path = Path(r'/Users/ppjaisri/Coding/phd/SLR_on_Git_VCS/data/filtered_papers_by_title.jsonl')

if filtered_papers_by_title_path.exists():
    filtered_papers_by_title_path.unlink()

all_papers_count = 0
removed_papers_count = {}
index = 1
for each_conference in all_semantic_scholar_conferences:
    with open(each_conference, 'r') as f:
        papers = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                papers.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    all_papers_count += len(papers)

    filtered_papers = []
    for paper in papers:
        title = paper.get('title', '')
        doi = paper.get('doi', '')
        abbr = paper.get('abbr', '')
        published_year = paper.get('published_year', None)

        print(f"Index {index}: Checking DOI: {doi}, Abbr: {abbr}, Published Year: {published_year}")
        index += 1

        # Remove papers with empty titles
        if not title:
            if 'remove_emty_title' not in removed_papers_count:
                removed_papers_count['remove_empty_title'] = 0
            removed_papers_count['remove_empty_title'] += 1
            continue

        # Remove papers with non-ASCII characters in the title
        if any(ord(char) > 127 for char in title):
            if 'remove_non_ascii_title' not in removed_papers_count:
                removed_papers_count['remove_non_ascii_title'] = 0
            removed_papers_count['remove_non_ascii_title'] += 1
            continue

        # Remove papers with titles that are too short (less than 5 characters)
        if len(title.strip()) < 5:
            if 'remove_short_title' not in removed_papers_count:
                removed_papers_count['remove_short_title'] = 0
            removed_papers_count['remove_short_title'] += 1
            continue

        # Remove papers with titles that are editorial content
        if title.strip().lower() in ['copyright', 'editorial board', 'editorial note', 'editorial', 'editorial preface', 'editorial introduction']:
            if 'remove_editorial_content' not in removed_papers_count:
                removed_papers_count['remove_editorial_content'] = 0
            removed_papers_count['remove_editorial_content'] += 1
            continue

        # Remove non-related papers based on keywords in the title
        bio_keywords = ['bio', 'biology', 'biological', 'biomedicine', 'biomedical', 'biochemistry', 'biophysics']
        if any(keyword in title.lower() for keyword in bio_keywords):
            if 'remove_bio_related' not in removed_papers_count:
                removed_papers_count['remove_bio_related'] = 0
            removed_papers_count['remove_bio_related'] += 1
            continue

        slr_keywords = ['systematic literature review', 'systematic review', 'literature review', 'survey', 'mapping study', 'scoping review']
        if any(keyword in title.lower() for keyword in slr_keywords):
            if 'remove_slr' not in removed_papers_count:
                removed_papers_count['remove_slr'] = 0
            removed_papers_count['remove_slr'] += 1
            continue

        survey_keywords = ['survey', 'review', 'overview']
        if any(keyword in title.lower() for keyword in survey_keywords):
            if 'remove_survey' not in removed_papers_count:
                removed_papers_count['remove_survey'] = 0
            removed_papers_count['remove_survey'] += 1
            continue

        filtered_papers.append(paper)

        with open(filtered_papers_by_title_path, 'a+') as f:
            # for paper in filtered_papers:
            json.dump(paper, f, ensure_ascii=False)
            f.write('\n')

print(f"Total papers removed: {json.dumps(removed_papers_count, indent=4)}")
print(f"Total papers collected after filtering: {all_papers_count - sum(removed_papers_count.values())}")


In [ ]:
filtered_papers_by_title_with__venue_variants_path = root_path.joinpath('data/filtered_papers_by_title_with_venue_variants.jsonl')


In [ ]:
def double_check_papers_venue_from_doi(doi: str, expected_venue: str, published_year: int) -> str:
    based_url = f'https://citation.doi.org/metadata/?doi={doi}'

    sleep(0.5)  # To avoid hitting the API rate limit
    res = requests.get(based_url, headers={'Accept': 'application/json'})
    if res.status_code != 200:
        print(f"Error: {res.status_code} - {res.text}")
        print(f"DOI: {doi}, Expected Venue: {expected_venue}, Published Year: {published_year}")
        return False

    data = res.json()
    container_title = data.get('container-title', None)

    if container_title:
        return container_title.lower()
    else:
        return None


In [ ]:
with open(filtered_papers_by_title_path, 'r') as f:
    filtered_by_title_papers = [json.loads(line) for line in f if line.strip()]

if filtered_papers_by_title_with__venue_variants_path.exists():
    filtered_papers_by_title_with__venue_variants_path.unlink()

papers_count = len(filtered_by_title_papers)
removed_papers_count = 0
index = 1

for paper in filtered_by_title_papers:
    title = paper.get('title', '')
    doi = paper.get('doi', '')
    abbr = paper.get('abbr', '')
    published_year = paper.get('published_year', None)

    print(f"Index {index}: Checking DOI: {doi}, Abbr: {abbr}, Published Year: {published_year}")
    index += 1

    if doi is None or doi == '':
        paper['venue_from_doi'] = None

    doi_venues = double_check_papers_venue_from_doi(doi, abbr, published_year)
    paper['venue_from_doi'] = doi_venues

    with open(filtered_papers_by_title_with__venue_variants_path, 'a+') as f:
        json.dump(paper, f, ensure_ascii=False)
        f.write('\n')


In [ ]:
with open(filtered_papers_by_title_with__venue_variants_path, 'r') as f:
    papers = [json.loads(line) for line in f if line.strip()]


In [ ]:
unique_venue_from_doi = set()
unique_venue = set()

for paper in papers:
    venue_from_doi = paper.get('venue_from_doi', None)
    venue = paper.get('venue', None)
    if venue_from_doi:
        unique_venue_from_doi.add(venue_from_doi)
    if venue:
        unique_venue.add(venue)

print(json.dumps(sorted(list(unique_venue_from_doi)), indent=4, ensure_ascii=False))
print(json.dumps(sorted(list(unique_venue)), indent=4, ensure_ascii=False))


In [ ]:
import re


In [ ]:
def parse_venue_name(raw_venue: str):
    text = " ".join(str(raw_venue).strip().split())
    lowered = text.lower()

    year_pat = r"\b(?:19|20)\d{2}\b|\b'\d{2}\b"
    order_pat = r"\b(?:\d{1,3}(?:st|nd|rd|th)|[ivxlcdm]+)\b"
    publisher_pat = r"\b(?:ieee|acm)(?:\s*/\s*(?:ieee|acm))?\b"

    # remove any prefix before first year/order
    year_m = re.search(year_pat, lowered)
    order_m = re.search(order_pat, lowered, flags=re.IGNORECASE)
    starts = [m.start() for m in [year_m, order_m] if m]
    if not starts:
        return {
            "original": text,
            "year": None,
            "publisher": None,
            "order": None,
            "name": text,
            "subtract": None,
            "abbr": None,
        }, "missing year/order"

    start_idx = min(starts)
    core = text[start_idx:]

    # extract abbreviation at the end: (...), e.g., (ICSE), (ICSE-SEET)
    abbr_m = re.search(r"\(([^()]{2,30})\)\s*$", core)
    abbr = None
    if abbr_m:
        raw_abbr = abbr_m.group(1).strip()
        # keep only the leading alphabetic chunk before the first non-alphabet character
        cleaned_m = re.match(r"[A-Za-z]+", raw_abbr)
        if cleaned_m and len(cleaned_m.group(0)) >= 2:
            abbr = cleaned_m.group(0)
            core = core[:abbr_m.start()].strip()
        else:
            abbr = None

    # split optional subtract by ":" or "-" (or en/em dash)
    parts = re.split(r"\s*:\s*|\s+[–—-]\s+", core, maxsplit=1)
    main_part = parts[0].strip()
    subtract = parts[1].strip() if len(parts) > 1 else None

    # extract mandatory components from main_part
    year_m2 = re.search(year_pat, main_part)
    order_m2 = re.search(order_pat, main_part, flags=re.IGNORECASE)
    publisher_m2 = re.search(publisher_pat, main_part, flags=re.IGNORECASE)

    year = int(year_m2.group(0)) if year_m2 else None
    order = order_m2.group(0).lower() if order_m2 else None
    publisher = publisher_m2.group(0).lower().replace(" ", "") if publisher_m2 else None

    # build "name" by removing year, publisher, order
    name = main_part
    name = re.sub(year_pat, "", name, count=1)
    name = re.sub(order_pat, "", name, count=1, flags=re.IGNORECASE)
    name = re.sub(publisher_pat, "", name, count=1, flags=re.IGNORECASE)
    name = re.sub(r"^\s*the\s+", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\s+", " ", name).strip(" ,.-").lower()
    name = name if name else None

    if all(v is None for v in [year, publisher, order, name, subtract, abbr]):
        return {
            "original": text,
            "year": None,
            "publisher": None,
            "order": None,
            "name": text,
            "subtract": subtract.lower() if subtract else None,
            "abbr": abbr.lower() if abbr else None,
        }, "missing all components"
    else:
        res = {
            "original": text,
            "year": year,
            "publisher": publisher,
            "order": order,
            "name": name,
            "subtract": subtract.lower() if subtract else None,
            "abbr": abbr.lower() if abbr else None,
        }

    return res, None


In [ ]:
source_venues = sorted(list(unique_venue_from_doi)) if "unique_venue_from_doi" in globals() else []
parsed_venues_from_doi = []

for venue_name in source_venues:
    parsed, reason = parse_venue_name(venue_name)
    parsed_venues_from_doi.append(parsed)

print(json.dumps(parsed_venues_from_doi, indent=4, ensure_ascii=False))


In [ ]:
parsed_venues_by_abbr = {}

for i in parsed_venues_from_doi:
    abbr = i.get('abbr', None)
    if abbr is None:
        if 'no_abbr' not in parsed_venues_by_abbr:
            parsed_venues_by_abbr['no_abbr'] = set()
        parsed_venues_by_abbr['no_abbr'].add(i['name'])
    if abbr not in parsed_venues_by_abbr:
        parsed_venues_by_abbr[abbr] = set()
    if i['name'] not in parsed_venues_by_abbr[abbr]:
        parsed_venues_by_abbr[abbr].add(i['name'])

parsed_venues_by_abbr['no_abbr'] = list(parsed_venues_by_abbr.get('no_abbr', []))
for abbr in parsed_venues_by_abbr:
    if abbr != 'no_abbr':
        parsed_venues_by_abbr[abbr] = list(parsed_venues_by_abbr[abbr])

print(json.dumps(parsed_venues_by_abbr, indent=4, ensure_ascii=False))


#### Filter by the venue collected from doi

In [ ]:
selected_venue = [
    'international conference on automated software engineering',
    'international conference on software analysis, evolution and reengineering',
    'international conference on software maintenance and evolution',
    'international conference on mining software repositories',
    'international conference on software engineering',
    'transactions on software engineering and methodology',
    'european software engineering conference and symposium on the foundations of software engineering',
    'automated software engineering',
    'journal of systems and software',
    'transactions on software engineering',
    'empirical software engineering',
    'Proc. ACM Softw. Eng.'
]


In [ ]:
with open(filtered_papers_by_title_with__venue_variants_path, 'r') as f:
    title_with_venue_variants_papers = [json.loads(line) for line in f if line.strip()]


In [34]:
filter_by_duble_check_from_doi_path = Path(r'/Users/ppjaisri/Coding/phd/SLR_on_Git_VCS/data/filter_by_double_check_from_doi.jsonl')


In [ ]:
if filter_by_duble_check_from_doi_path.exists():
    filter_by_duble_check_from_doi_path.unlink()

# removed_papers_count = 0
selected_papers_count = 0
for paper in title_with_venue_variants_papers:
    venue = paper.get('venue', None)
    venue_from_doi = paper.get('venue_from_doi', None)
    if venue_from_doi:
        for selected in selected_venue:
            if selected in venue_from_doi:

                with open(filter_by_duble_check_from_doi_path, 'a+') as f:
                    json.dump(paper, f, ensure_ascii=False)
                    f.write('\n')

                selected_papers_count += 1
                break
            elif venue == 'Proc. ACM Softw. Eng.':

                with open(filter_by_duble_check_from_doi_path, 'a+') as f:
                    json.dump(paper, f, ensure_ascii=False)
                    f.write('\n')

                selected_papers_count += 1
                break
            
            continue

print(f'Papers removed: {len(title_with_venue_variants_papers) - selected_papers_count}')


#### Select only empirical studies

In [ ]:
filter_empirical_study_path = root_path.joinpath('data/filter_empirical_study.jsonl')


In [35]:
with open(filter_by_duble_check_from_doi_path, 'r') as f:
    double_checked_from_doi_papers = [json.loads(line) for line in f if line.strip()]
    

In [ ]:
papers_per_year_by_venue = {}

for paper in double_checked_from_doi_papers:
    published_year = paper.get('published_year', None)
    abbr = paper.get('abbr', None)
    venue = paper.get('venue', None)

    if venue == 'Proc. ACM Softw. Eng.':
        abbr = 'FSE'

    if any([published_year is None, abbr is None]):
        continue

    if abbr not in papers_per_year_by_venue:
        papers_per_year_by_venue[abbr] = {}
    if 'published_year' not in papers_per_year_by_venue[abbr]:
        papers_per_year_by_venue[abbr]['published_year'] = {}
    if published_year not in papers_per_year_by_venue[abbr]['published_year']:
        papers_per_year_by_venue[abbr]['published_year'][published_year] = 0

    papers_per_year_by_venue[abbr]['published_year'][published_year] += 1
    papers_per_year_by_venue[abbr]['sum'] = papers_per_year_by_venue[abbr].get('sum', 0) + 1
    
papers_per_year_by_venue['all_papers'] = sum([papers_per_year_by_venue[abbr]['sum'] for abbr in papers_per_year_by_venue])


In [ ]:
for abbr, data in papers_per_year_by_venue.items():
    if not isinstance(data, dict):
        continue
    published_years = data.get('published_year', {})
    sorted_published_years = dict(sorted(published_years.items()))
    papers_per_year_by_venue[abbr]['published_year'] = sorted_published_years

print(json.dumps(papers_per_year_by_venue, indent=4, ensure_ascii=False))


In [ ]:
double_checked_from_doi_papers


In [36]:
empirical_keywords = [
    'empirical',
    'empirically',
    'empirical study',
    'empirical studies',
    'empirical analysis',
    'empirical evaluation',
    'empirical investigation',
    'empirical assessment',
    'empirical evidence',
]


In [37]:
if filter_empirical_study_path.exists():
    filter_empirical_study_path.unlink()


index = 1
removed_papers_count = 0
papers_count = len(double_checked_from_doi_papers)
for paper in double_checked_from_doi_papers:
    title = paper.get('title', '')
    doi = paper.get('doi', '')
    abbr = paper.get('abbr', '')
    published_year = paper.get('published_year', None)
    venue = paper.get('venue', None)

    if venue == 'Proc. ACM Softw. Eng.':
        abbr = 'FSE'

    print(f"Index {index}: Checking DOI: {doi}, Abbr: {abbr}, Published Year: {published_year}")
    index += 1

    # Remove non-empirical studies based on keywords in the title and abstract
    if not any(keyword in title.lower() for keyword in empirical_keywords):
        removed_papers_count += 1
        continue

    with open(filter_empirical_study_path, 'a+') as f:
        json.dump(paper, f, ensure_ascii=False)
        f.write('\n')


Index 1: Checking DOI: 10.1016/j.jss.2023.111724, Abbr: JSS, Published Year: 2022
Index 2: Checking DOI: 10.1016/j.jss.2022.111429, Abbr: JSS, Published Year: 2022
Index 3: Checking DOI: 10.1016/j.jss.2021.110985, Abbr: JSS, Published Year: 2021
Index 4: Checking DOI: 10.1016/j.jss.2025.112691, Abbr: JSS, Published Year: 2025
Index 5: Checking DOI: 10.1016/j.jss.2023.111732, Abbr: JSS, Published Year: 2023
Index 6: Checking DOI: 10.1016/j.jss.2022.111269, Abbr: JSS, Published Year: 2022
Index 7: Checking DOI: 10.1016/j.jss.2022.111561, Abbr: JSS, Published Year: 2022
Index 8: Checking DOI: 10.1016/j.jss.2025.112345, Abbr: JSS, Published Year: 2025
Index 9: Checking DOI: 10.1016/j.jss.2024.112121, Abbr: JSS, Published Year: 2024
Index 10: Checking DOI: 10.1016/j.jss.2024.112247, Abbr: JSS, Published Year: 2024
Index 11: Checking DOI: 10.1016/j.jss.2022.111385, Abbr: JSS, Published Year: 2022
Index 12: Checking DOI: 10.1016/j.jss.2025.112697, Abbr: JSS, Published Year: 2025
Index 13: Che

In [38]:
with open(filter_empirical_study_path, 'r') as f:
    filter_empirical_papers = [json.loads(line) for line in f if line.strip()]


In [39]:
empirical_papers_per_year_by_venue = {}

for paper in filter_empirical_papers:
    published_year = paper.get('published_year', None)
    abbr = paper.get('abbr', None)
    venue = paper.get('venue', None)

    if venue == 'Proc. ACM Softw. Eng.':
        abbr = 'FSE'

    if any([published_year is None, abbr is None]):
        continue

    if abbr not in empirical_papers_per_year_by_venue:
        empirical_papers_per_year_by_venue[abbr] = {}
    if 'published_year' not in empirical_papers_per_year_by_venue[abbr]:
        empirical_papers_per_year_by_venue[abbr]['published_year'] = {}
    if published_year not in empirical_papers_per_year_by_venue[abbr]['published_year']:
        empirical_papers_per_year_by_venue[abbr]['published_year'][published_year] = 0

    empirical_papers_per_year_by_venue[abbr]['published_year'][published_year] += 1
    empirical_papers_per_year_by_venue[abbr]['sum'] = empirical_papers_per_year_by_venue[abbr].get('sum', 0) + 1
    
empirical_papers_per_year_by_venue['all_papers'] = sum([empirical_papers_per_year_by_venue[abbr]['sum'] for abbr in empirical_papers_per_year_by_venue])


In [40]:
for key in empirical_papers_per_year_by_venue.keys():
    print(key)


JSS
SANER
FSE
EMSE
ICSE
TOSEM
TSE
ASE
ICSME
MSR
all_papers


In [41]:
for abbr, data in empirical_papers_per_year_by_venue.items():
    if not isinstance(data, dict):
        continue
    published_years = data.get('published_year', {})
    sorted_published_years = dict(sorted(published_years.items()))
    empirical_papers_per_year_by_venue[abbr]['published_year'] = sorted_published_years

print(json.dumps(empirical_papers_per_year_by_venue, indent=4, ensure_ascii=False))


{
    "JSS": {
        "published_year": {
            "2021": 13,
            "2022": 8,
            "2023": 8,
            "2024": 13,
            "2025": 12
        },
        "sum": 54
    },
    "SANER": {
        "published_year": {
            "2021": 6,
            "2022": 6,
            "2023": 6,
            "2024": 8,
            "2025": 7
        },
        "sum": 33
    },
    "FSE": {
        "published_year": {
            "2021": 5,
            "2022": 6,
            "2023": 3,
            "2024": 4,
            "2025": 8
        },
        "sum": 26
    },
    "EMSE": {
        "published_year": {
            "2021": 21,
            "2022": 14,
            "2023": 26,
            "2024": 26,
            "2025": 20
        },
        "sum": 107
    },
    "ICSE": {
        "published_year": {
            "2021": 6,
            "2022": 2,
            "2023": 15,
            "2024": 8,
            "2025": 6
        },
        "sum": 37
    },
    "TOSEM": {
        "publi

## Collect abstract of the empirical studies

In [ ]:
empirical_papers_with_abstract_path = root_path.joinpath('data/empirical_papers_with_abstract.jsonl')


In [44]:
with open(filter_empirical_study_path, 'r') as f:
    filter_empirical_papers = [json.loads(line) for line in f if line.strip()]


In [ ]:
semantic_scholar_api_key = 'YOUR_API_KEY_HERE'  # Replace with your actual


In [ ]:
def retrieve_abstract_from_semantic_scholar(doi: str) -> tuple:
    headers = {
        'Accept': 'application/json',
        'x-api-key': semantic_scholar_api_key
    }
    base_url = f'https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}?fields=title,abstract'
    sleep(1.5)  # To avoid hitting the API rate limit
    res = requests.get(base_url, headers=headers)
    if res.status_code != 200:
        print(f"Error: {res.status_code} - {res.text}")
        print(f"DOI: {doi}")
        return None, None

    data = res.json()
    title = data.get('title', None)
    abstract = data.get('abstract', None)
    return title, abstract


In [46]:
if empirical_papers_with_abstract_path.exists():
    empirical_papers_with_abstract_path.unlink()

index = 1
for paper in filter_empirical_papers:
    doi = paper.get('doi', None)
    if doi is None or doi == '':
        continue

    print(f"Index {index}: Retrieving abstract for DOI: {doi}")
    title, abstract = retrieve_abstract_from_semantic_scholar(doi)

    paper['abstract'] = abstract

    with open(empirical_papers_with_abstract_path, 'a+') as f:
        json.dump(paper, f, ensure_ascii=False)
        f.write('\n')


Index 1: Retrieving abstract for DOI: 10.1016/j.jss.2022.111269
Index 1: Retrieving abstract for DOI: 10.1016/j.jss.2022.111561
Error: 429 - {"message": "Too Many Requests. Please wait and try again or apply for a key for higher rate limits. https://www.semanticscholar.org/product/api#api-key-form", "code": "429"}
DOI: 10.1016/j.jss.2022.111561
Index 1: Retrieving abstract for DOI: 10.1016/j.jss.2025.112601
Index 1: Retrieving abstract for DOI: 10.1016/j.jss.2023.111684
Error: 429 - {"message": "Too Many Requests. Please wait and try again or apply for a key for higher rate limits. https://www.semanticscholar.org/product/api#api-key-form", "code": "429"}
DOI: 10.1016/j.jss.2023.111684
Index 1: Retrieving abstract for DOI: 10.1016/j.jss.2023.111787
Error: 429 - {"message": "Too Many Requests. Please wait and try again or apply for a key for higher rate limits. https://www.semanticscholar.org/product/api#api-key-form", "code": "429"}
DOI: 10.1016/j.jss.2023.111787
Index 1: Retrieving abs

In [47]:
with open(empirical_papers_with_abstract_path, 'r') as f:
    empirical_papers_with_abstract = [json.loads(line) for line in f if line.strip()]


In [48]:
journal_papers = ['tse', 'tosem', 'emse', 'jss']
conference_papers = ['icse', 'fse', 'ase', 'icsme', 'saner', 'msr']


In [ ]:
paper_with_abstract_count = {
    'journal_papers': {},
    'conference_papers': {},
    'with_abstract': 0,
    'without_abstract': 0
}

paper_with_abstract_count_by_year = {}

print(f"All papers: {len(empirical_papers_with_abstract)}")

for paper in empirical_papers_with_abstract:
    abbr = paper.get('abbr', None)
    abstract = paper.get('abstract', None)
    published_year = paper.get('published_year', None)

    if abbr is None:
        continue

    # print(f'Abbr is {abbr}')
    isJournal = abbr.lower() in journal_papers
    isJournal_category = 'journal_papers' if isJournal else 'conference_papers'
    # print(f'Category is {isJournal_category}')


    if abbr not in paper_with_abstract_count[isJournal_category]:
        paper_with_abstract_count[isJournal_category][abbr] = {'with_abstract': 0, 'without_abstract': 0}

    if abstract is not None and abstract.strip() != '':
        # print(f"Abstract found for {abbr}")
        paper_with_abstract_count[isJournal_category][abbr]['with_abstract'] += 1
        paper_with_abstract_count['with_abstract'] += 1

        paper_with_abstract_count_by_year.setdefault(abbr, {})
        paper_with_abstract_count_by_year[abbr].setdefault(published_year, 0)
        paper_with_abstract_count_by_year[abbr][published_year] += 1   
    else:
        # print(f"No abstract found for {abbr}")
        paper_with_abstract_count[isJournal_category][abbr]['without_abstract'] += 1
        paper_with_abstract_count['without_abstract'] += 1

paper_with_abstract_count_by_year = {abbr: dict(sorted(years.items())) for abbr, years in paper_with_abstract_count_by_year.items()}


All papers: 447


In [50]:
print(json.dumps(paper_with_abstract_count, indent=4, ensure_ascii=False))


{
    "journal_papers": {
        "JSS": {
            "with_abstract": 11,
            "without_abstract": 43
        },
        "EMSE": {
            "with_abstract": 46,
            "without_abstract": 61
        },
        "TOSEM": {
            "with_abstract": 32,
            "without_abstract": 18
        },
        "TSE": {
            "with_abstract": 23,
            "without_abstract": 24
        }
    },
    "conference_papers": {
        "SANER": {
            "with_abstract": 19,
            "without_abstract": 14
        },
        "FSE": {
            "with_abstract": 11,
            "without_abstract": 15
        },
        "ICSE": {
            "with_abstract": 15,
            "without_abstract": 22
        },
        "ASE": {
            "with_abstract": 12,
            "without_abstract": 24
        },
        "ICSME": {
            "with_abstract": 10,
            "without_abstract": 8
        },
        "MSR": {
            "with_abstract": 23,
            "without

In [91]:
print(json.dumps(paper_with_abstract_count_by_year, indent=4, ensure_ascii=False))


{
    "JSS": {
        "2021": 6,
        "2022": 2,
        "2023": 1,
        "2024": 1,
        "2025": 1
    },
    "SANER": {
        "2021": 3,
        "2022": 4,
        "2023": 4,
        "2024": 3,
        "2025": 5
    },
    "FSE": {
        "2021": 1,
        "2022": 1,
        "2024": 4,
        "2025": 5
    },
    "EMSE": {
        "2021": 8,
        "2022": 4,
        "2023": 11,
        "2024": 13,
        "2025": 10
    },
    "ICSE": {
        "2021": 3,
        "2023": 9,
        "2024": 1,
        "2025": 2
    },
    "TOSEM": {
        "2021": 2,
        "2022": 3,
        "2023": 7,
        "2024": 11,
        "2025": 9
    },
    "TSE": {
        "2021": 3,
        "2022": 6,
        "2023": 5,
        "2024": 6,
        "2025": 3
    },
    "ASE": {
        "2021": 3,
        "2022": 1,
        "2023": 4,
        "2025": 4
    },
    "ICSME": {
        "2022": 3,
        "2023": 2,
        "2024": 1,
        "2025": 4
    },
    "MSR": {
        "2021": 3,
    

#### Random sample
- Confidence Level: 95%
- Margin of Error: 5%
- Population Proportion: 50%
- Population Size: 917 papers

In [18]:
import math

population_size = len(classified_papers) - 1

confidence_level = 0.95
margin_of_error = 0.05
p = 0.5

z_score = 1.96  # 95% confidence

# Cochran's formula for infinite population
n0 = (z_score**2 * p * (1 - p)) / (margin_of_error**2)

# Finite population correction
sample_size = math.ceil(n0 / (1 + ((n0 - 1) / population_size)))

print(f"Population size: {population_size}")
print(f"Required sample size at {confidence_level * 100:.0f}% confidence and {margin_of_error * 100:.0f}% margin of error: {sample_size}")


Population size: 916
Required sample size at 95% confidence and 5% margin of error: 271


In [22]:
sample_papers_json_path = Path("/Users/ppjaisri/Coding/phd/SLR_on_Git_VCS/data/sample_papers.jsonl")
sample_papers_csv_path = Path("/Users/ppjaisri/Coding/phd/SLR_on_Git_VCS/data/sample_papers.csv")

if sample_papers_json_path.exists():
    sample_papers_json_path.unlink()

if sample_papers_csv_path.exists():
    sample_papers_csv_path.unlink()

sampled_papers = np.random.choice(classified_papers, size=min(271, len(classified_papers)), replace=False).tolist()

with open(sample_papers_json_path, "w") as f:
    for paper in sampled_papers:
        json.dump(paper, f, ensure_ascii=False)
        f.write("\n")

with open(sample_papers_csv_path, "w") as f:
    f.write('title,link,software_engineering_field,provides_dataset,dataset_names,analysis_type,1st_research_category,2nd_research_category,3rd_research_category,notes\n')
    for paper in sampled_papers:
        title = paper['title']
        link = paper.get('link', '')  # Use an empty string if 'link' is not present

        classification = paper.get('llm_classification', None)
        if classification:
            software_engineering_field = classification.get('software_engineering_field', None)
            provides_dataset = classification.get('provides_dataset', None)
            dataset_names = classification.get('dataset_names', None)

            if dataset_names is not None:
                dataset_names = ', '.join(dataset_names) if isinstance(dataset_names, list) else str(dataset_names)
            else:
                dataset_names = ''

            analysis_type = classification.get('analysis_type', '')

            research_categories = classification.get('research_categories', None)
            research_categories = [category['name'] for category in research_categories if category] if research_categories else None

            notes = classification.get('notes', None)

            # Handle the case where research_categories is a list of dictionaries
            if isinstance(research_categories, list):
                research_categories = [
                    cat.get('name') if isinstance(cat, dict) else cat
                    for cat in research_categories
                    if cat
                ]

            lines = f'"{title}","{link}","{software_engineering_field}","{provides_dataset}","{dataset_names}","{analysis_type}","{research_categories[0] if len(research_categories) > 0 else ''}","{research_categories[1] if len(research_categories) > 1 else ''}","{research_categories[2] if len(research_categories) > 2 else ''}","{notes}"\n'

        f.write(lines)

print(f"Saved {len(sampled_papers)} papers to {sample_papers_json_path}")
print(f"Saved {len(sampled_papers)} papers to {sample_papers_csv_path}")


Saved 271 papers to /Users/ppjaisri/Coding/phd/SLR_on_Git_VCS/data/sample_papers.jsonl
Saved 271 papers to /Users/ppjaisri/Coding/phd/SLR_on_Git_VCS/data/sample_papers.csv


#### Classify by keywords

In [69]:
# current_sample_path = Path("/Users/ppjaisri/Coding/phd/SLR_on_Git_VCS/data/current_sample_papers.jsonl")
keywords_classified_papers_json_path = Path("/Users/ppjaisri/Coding/phd/SLR_on_Git_VCS/data/keywords_classified_papers.jsonl")
keywords_classified_papers_csv_path = Path("/Users/ppjaisri/Coding/phd/SLR_on_Git_VCS/data/keywords_classified_papers.csv")


In [68]:
# with open(current_sample_path, "r") as f:
#     current_sample_papers = [json.loads(line) for line in f if line.strip()]

with open(llm_classified_papers_path, "r") as f:
    current_sample_papers = [json.loads(line) for line in f if line.strip()]


In [66]:
qualitative_keywords = [
    "qualitative", "qualitative study", "qualitative analysis", "qualitative research",
    "interview", "interviews", "interview-based",
    "thematic", "thematic analysis", "thematic coding",
    "coding", "coded", "coding scheme", "coding process",
    "classification", "categorized", "categorization", "taxonomy", "taxonomy-based",
    "grounded theory", "content analysis", "open coding", "axial coding", "selective coding",
    "focus group", "focus groups", "focus-group",
    "case study", "case studies", "case-study",
    "ethnography", "ethnographic", "participant observation",
    "narrative analysis", "phenomenology", "phenomenological",
    "discourse analysis",
    "member checking", "reflexivity", "saturation", "theoretical saturation",
    "inductive reasoning", "inductive", "deductive reasoning", "deductive",
    "exploratory study", "exploratory analysis", "descriptive study",
    "survey interview", "manual analysis", "manual inspection", "human analysis"
]

quantitative_keywords = [
    "quantitative", "quantitative study", "quantitative analysis", "quantitative evaluation",
    "numerical", "numeric", "statistics", "statistical", "statistically",
    "experiment", "experiments", "experimental", "controlled experiment", "randomized experiment",
    "percentage", "percent", "%", "mean", "means", "median", "mode",
    "variance", "standard deviation", "std", "std dev", "confidence interval",
    "measurement", "measurements", "metric", "metrics", "measure", "measured",
    "benchmark", "benchmarking", "benchmark dataset", "benchmark suite",
    "large-scale evaluation", "large scale evaluation", "large-scale study", "large scale study",
    "empirical evaluation", "empirical study", "empirical analysis", "empirical investigation",
    "ablation study", "performance evaluation", "comparative evaluation",
    "correlation", "regression", "significance", "p-value", "statistical test", "hypothesis test",
    "sample size", "dataset size", "effect size", "precision", "recall", "f1-score",
    "accuracy", "error rate", "false positive", "false negative", "true positive", "true negative",
    "precision-recall", "precision", "recall", "roc curve", "confusion matrix", "anova", "t-test", "chi-square test",
]

both_qualitative_and_quantitative_keywords = [
    "qualitative and quantitative",
    "quantitative and qualitative",
    "qualitative/quantitative",
    "quantitative/qualitative",
    "qualitative-quantitative",
    "quantitative-qualitative",
    "mixed methods",
    "mixed-methods",
    "mixed method",
    "mixed-method",
    "mixed methods study",
    "mixed-methods study",
    "mixed method study",
    "mixed-method study",
    "mixed methods research",
    "mixed-methods research",
    "mixed method research",
    "mixed-method research",
    "mixed methods analysis",
    "mixed-methods analysis",
    "mixed method analysis",
    "mixed-method analysis",
    "mixed approaches",
    "multi-method",
    "multi methods",
    "multi-methods",
    "multi-method study",
    "multi-methods study",
    "mixed-approach",
    "triangulated study",
    "triangulation study",
    "triangulation",
    "method triangulation",
    "data triangulation",
    "survey and interview",
    "survey interviews",
    "interview and survey",
    "interviews and surveys",
    "qualitative survey",
    "quantitative survey",
    "case survey",
    "case study and experiment",
    "case studies and experiments",
    "qualitative evaluation and quantitative evaluation",
    "mixed approach",
    "mixed approaches study",
    "mixed-approaches",
    "mixed-approach study",
    "mixed-approaches study",
    "mixed approach study",
    "mixed approach research",
    "mixed approaches research",
    "mixed-approach research",
    "mixed-approaches research",
    "mixed-approach analysis",
    "mixed-approaches analysis"
]


In [70]:
if keywords_classified_papers_json_path.exists():
    keywords_classified_papers_json_path.unlink()

if keywords_classified_papers_csv_path.exists():
    keywords_classified_papers_csv_path.unlink()

for paper in current_sample_papers:
    title = paper['title']
    abstract = paper.get('abstract', '')

    test_string = f"{title}_{abstract}".lower()
    paper['analysis_explicit_mention'] = True

    found_both = False

    found_both = any(keyword in test_string for keyword in both_qualitative_and_quantitative_keywords)
    found_quantitative = any(keyword in test_string for keyword in quantitative_keywords)
    found_qualitative = any(keyword in test_string for keyword in qualitative_keywords)

    if found_both or (found_quantitative and found_qualitative):
        paper['keyword_analysis_type'] = 'both'
    elif found_quantitative:
        paper['keyword_analysis_type'] = 'quantitative'
    elif found_qualitative:
        paper['keyword_analysis_type'] = 'qualitative'
    else:
        paper['keyword_analysis_type'] = ''
        paper['analysis_explicit_mention'] = False

    with open(keywords_classified_papers_json_path, "a+") as f:
        json.dump(paper, f, ensure_ascii=False)
        f.write("\n")

with open(keywords_classified_papers_csv_path, "w") as f:
    # f.write('title,link,software_engineering_field,provides_dataset,dataset_names,analysis_type,1st_research_category,2nd_research_category,3rd_research_category,notes\n')
    f.write('title,link,software_engineering_field,provides_dataset,dataset_names,llm_classification_analysis_type,research_category,notes,analysis_explicit_mention,keyword_analysis_type\n')
    for paper in current_sample_papers:
        title = paper['title']
        link = paper.get('link', '')  # Use an empty string if 'link' is not present

        classification = paper.get('llm_classification', None)
        if classification:
            software_engineering_field = classification.get('software_engineering_field', None)
            provides_dataset = classification.get('provides_dataset', None)
            dataset_names = classification.get('dataset_names', None)

            if dataset_names is not None:
                dataset_names = ', '.join(dataset_names) if isinstance(dataset_names, list) else str(dataset_names)
            else:
                dataset_names = ''

            llm_analysis_type = classification.get('analysis_type', '')
            explicit_mention = paper.get('analysis_explicit_mention', False)
            analysis_type = paper.get('keyword_analysis_type', '')

            research_categories = classification.get('research_categories', None)
            research_categories = [category['name'] for category in research_categories if category] if research_categories else None

            notes = classification.get('notes', None)

            # Handle the case where research_categories is a list of dictionaries
            if isinstance(research_categories, list):
                research_categories = [
                    cat.get('name') if isinstance(cat, dict) else cat
                    for cat in research_categories
                    if cat
                ]

            lines = f'"{title}","{link}","{software_engineering_field}","{provides_dataset}","{dataset_names}","{llm_analysis_type}","{research_categories[0] if len(research_categories) > 0 else ''}","{notes}","{explicit_mention}","{analysis_type}"\n'

        f.write(lines)


In [92]:
current_sample_papers


[{'title': 'An empirical study of CGO usage in Go projects - Distribution, purposes, patterns and critical issues',
  'doi': '10.1016/j.jss.2025.112601',
  'link': 'https://doi.org/10.1016/j.jss.2025.112601',
  'published_year': 2025,
  'venue': 'Journal of Systems and Software',
  'abbr': 'JSS',
  'citation_count': 0,
  'venue_from_doi': 'journal of systems and software',
  'abstract': 'Multilingual software development integrates multiple languages into a single application, with the Foreign Function Interface (FFI) enabling seamless interaction. While FFI boosts efficiency and extensibility, it also introduces risks. Existing studies focus on FFIs in languages like Python and Java, neglecting CGO, the emerging FFI in Go, which poses unique risks. To address these concerns, we conduct an empirical study of CGO usage across 920 open-source Go projects. Our study aims to reveal the distribution, patterns, purposes, and critical issues associated with CGO, offering insights for develope

In [94]:
for paper in current_sample_papers:
    year = paper.get('published_year', None)
    abbr = paper.get('abbr', None)

    print(abbr)


JSS
JSS
JSS
JSS
JSS
JSS
JSS
JSS
JSS
JSS
JSS
SANER
SANER
SANER
SANER
SANER
SANER
SANER
SANER
SANER
SANER
SANER
SANER
SANER
SANER
SANER
SANER
SANER
SANER
SANER
FSE
FSE
FSE
FSE
FSE
FSE
FSE
FSE
FSE
FSE
FSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
EMSE
ICSE
ICSE
ICSE
ICSE
ICSE
ICSE
ICSE
ICSE
ICSE
ICSE
ICSE
ICSE
ICSE
ICSE
ICSE
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TOSEM
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
TSE
ASE
ASE
ASE
ASE
ASE
ASE
ASE
ASE
ASE
ASE
ASE
ASE
ICSME
ICSME
ICSME
ICSME
ICSME
ICSME
ICSME
ICSME
ICSME
ICSME
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
MSR
